In [1]:
# https://miro.medium.com/v2/resize:fit:1400/0*9G8VvkAZUNv-vQYr.png

In [2]:
# RAG system using LangChain and OpenAI

In [3]:
# Colab setup
# !pip install -U \
# langchain==0.3.27 \
# langchain-core==0.3.79 \
# langchain-community==0.3.31 \
# langchain-openai==0.3.35 \
# langchain-text-splitters==0.3.9 \
# langchainhub \
# faiss-cpu \
# sentence-transformers -q

In [4]:
# “Let me ask you something — how many of you have used ChatGPT or Gemini or Copilot and realized that sometimes it gives beautifully
# structured answers, but... they’re confidently wrong?”
# Large Language Models, or LLMs, are like very knowledgeable storytellers. They’re trained on massive text data from the
# internet, but once the training is complete, they stop learning. They generate responses based on patterns they’ve
# seen, but they don’t have real-time access to new information or the ability to verify facts. This means that while
# they can produce impressive and coherent answers, they can also confidently provide incorrect or outdated information.
# It’s important to approach their responses with a critical eye and verify facts when necessary.

# They don’t know your private data, your company’s documents, or anything that happened after their cutoff date.
# So, when you ask a question beyond their memory, they do what humans often do under pressure — they guess elegantly.
# They create a plausible-sounding answer based on the patterns they’ve learned, but it might not be accurate.
# This is why it’s crucial to use LLMs as tools for generating ideas or providing general information, rather than
# relying on them for critical or up-to-date facts without verification.

# This guessing problem is what we call ‘hallucination.’ And that’s not acceptable in domains like healthcare, banking, or
# legal — where accuracy matters more than eloquence.
# That’s where RAG — Retrieval-Augmented Generation — comes in. It’s like giving your LLM a search engine for truth before
# it starts speaking.

In [5]:
# Q: Imagine you’re an interviewer who’s about to take a technical interview. You haven’t prepared in months. Would you rather:
# A) Trust your memory from last year’s prep, or
# B) Quickly Google the latest concepts before answering?
# Ans: B) Quickly Google the latest concepts before answering.
# That’s exactly what RAG does.
# Instead of relying only on the model’s old ‘memory,’ RAG lets the model retrieve the most recent and relevant information
# first — and then generate an answer using both what it already knows and what it just fetched.

In [6]:
# Q: Suppose your company wants a chatbot that understands your own documents, policies, and reports.
# Would you rather:
# A) Train or fine-tune your own LLM with all that data,
# or
# B) Plug your data into ChatGPT or an open-source model that already exists and just let it ‘refer’ to it when needed?
# Fine-tuning or retraining a large language model is extremely expensive.
# Let’s say you’re using a 13-billion parameter model — fine-tuning it on your data could cost thousands of dollars, not
# just for one-time training, but also for storage, GPUs, and re-training every time your data changes.

| Factor           | Fine-Tuning Approach             | RAG Approach               |
| :--------------- | :------------------------------- | :------------------------- |
| **Cost**         | Very high (compute + retraining) | Low (no model training)    |
| **Time**         | Weeks or months                  | Hours or days              |
| **Data Updates** | Needs retraining                 | Instantly reflected        |
| **Flexibility**  | Fixed to trained data            | Dynamic and updatable      |
| **Risk**         | Possible overfitting             | None (data stays external) |


In [7]:
# So if your HR policy or customer data changes next week,
# in fine-tuning, you’d have to train again, costing you time and GPU cycles.
# But with RAG, you just update your vector database.
# The model stays the same — it simply fetches new facts before answering.

In [8]:
# Fine-tuning is like engraving information on stone — costly and hard to change.

# RAG is like reading from a dynamic library — cheaper, flexible, and always current.

In [9]:
# Say you have 10,000 internal documents and you want a chatbot that can answer from them.

# | Step           | Fine-Tuning Cost (Approx.) | RAG Cost (Approx.)               |
# |----------------|----------------------------|----------------------------------|
# | Model Training | $3,000–$10,000             | $0                               |
# | GPU Hosting    | $1,000 / month             | $100–$200 / month (Vector DB)    |
# | Data Update    | Retraining again ($3K+)    | Re-embedding ($50–$100)          |
# | Total (Year 1) | ~$20,000+                  | ~$1,000–$2,000                   |

In [10]:
# RAG VS Fine-Tuning: similar to comparing: Whether to go for Petrol or Diesel car?

In [11]:
# https://miro.medium.com/v2/resize:fit:1400/0*9G8VvkAZUNv-vQYr.png

## **Problem Statement**

In real-world applications, organizations often store important information in documents such as **policy manuals, handbooks, or technical PDFs**. Traditional language models like GPT-4 do not have access to these documents unless explicitly given the data.

This project demonstrates how to build a **Retrieval-Augmented Generation (RAG)** system using LangChain and OpenAI. It allows the language model to **search inside a PDF document**, retrieve relevant information, and generate accurate, context-specific answers.

---

## **Objectives**

1. Load content from a PDF file into a LangChain-compatible format.
2. Split the PDF content into manageable chunks for efficient vector storage.
3. Generate embeddings for each chunk using OpenAI Embeddings.
4. Store and search the chunks using the FAISS vector store.
5. Use GPT-4 Turbo as the language model to generate context-aware answers using retrieved chunks.
6. Ensure that the implementation uses the latest LangChain and OpenAI APIs without any deprecation warnings.

---

## **Expected Outcomes**

- A working RAG pipeline that uses PDF content as a knowledge base.
- The system can answer user queries based on the information stored inside the PDF.
- It will retrieve relevant passages from the PDF and use them to generate answers via GPT-4.
- The setup will be modular, extensible, and aligned with the latest LangChain version.

---

In [12]:
import os
import sys
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings, ChatOpenAI

from langchain_core.vectorstores import InMemoryVectorStore

# from langchain.chains import RetrievalQA
from langchain_classic.chains import RetrievalQA

In [13]:
# Production-readiness utilities: prompt control, JSON logging, timing and evaluation support
import json
import time
import uuid
import math
from pathlib import Path
from datetime import datetime, timezone
from statistics import mean

# from langchain.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

print(f"Notebook Python executable: {sys.executable}")

Notebook Python executable: /opt/homebrew/opt/python@3.11/bin/python3.11


In [14]:
# Step 1: Load environment variables (.env should have your OPENAI_API_KEY)
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError(
        "OPENAI_API_KEY was not found. Add it to a .env file locally or load it from Colab userdata before running the RAG cells."
    )

print("OPENAI_API_KEY loaded successfully.")


OPENAI_API_KEY loaded successfully.


In [16]:
# !pip install pypdf -q

In [15]:
# Step 2: Load PDF document
# # https://python.langchain.com/docs/integrations/document_loaders/

pdf_path = "docs/company_policy.pdf"
if not Path(pdf_path).is_file():
    raise FileNotFoundError(f"PDF file not found at path: {pdf_path}")

loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(f"Loaded {len(documents)} pages from the PDF.")

Loaded 24 pages from the PDF.


In [18]:
# documents

In [19]:
# Lets print the first page to see what it looks like
print("First page content:")
print(documents[0].page_content)

First page content:
SPIL Corporate HR Policies  
 
 
SIRCA PAINTS INDIA LTD 
NEW DELHI  
 
 
 
 
CORPORATE  
  HUMAN RESOURCES 
POLICIES & MANUALS


In [20]:
# 1. PDF Loaders
# Loader =>	Use When...	=> Handles Images/Tables?	=> Code Example
# PyPDFLoader	Simple text-based PDFs    => No	=> loader = PyPDFLoader("path/to/pdf")
# PDFMinerLoader	Needs layout-aware text extraction => No	=>	loader = PDFMinerLoader("file.pdf")
# PDFPlumberLoader	=> Needs table extraction => Basic tables	=> loader = PDFPlumberLoader("file.pdf")
# UnstructuredPDFLoader	=> Complex structure, mixed text, tables, images => Yes (OCR + layout)    => loader = UnstructuredPDFLoader("file.pdf")
# PyMuPDFLoader	Text + metadata (fast, accurate) => (Limited)	=> loader = PyMuPDFLoader("file.pdf")

# 2. Word Documents (DOC / DOCX)
# Loader	=> Use When...	=> Code
# UnstructuredWordDocumentLoader	You have .docx files	=> loader = UnstructuredWordDocumentLoader("file.docx")
# Docx2txtLoader	Basic text extraction from .docx	=> loader = Docx2txtLoader("file.docx")

# 3. Excel Files (XLS / XLSX)
# Loader	Use When...	Code
# UnstructuredExcelLoader	Full sheet extraction	=> loader = UnstructuredExcelLoader("file.xlsx")
# PandasExcelLoader	Structured loading using pandas	=> loader = PandasExcelLoader("file.xlsx")

# 4. CSV Files
# Loader	Use When...	Code
# CSVLoader	Simple CSV reading	=> loader = CSVLoader("file.csv")
# PandasCSVLoader	Use Pandas for control & filtering	=> loader = PandasCSVLoader("file.csv")

# 5. JSON / JSONL
# Loader	Use When...	Code
# JSONLoader	Simple .json file with nested fields    => loader = JSONLoader(file_path="file.json", jq_schema='.data[].text', text_content=False)
# JSONLinesLoader	Line-delimited JSON (JSONL)	=> loader = JSONLinesLoader("file.jsonl")

# 6. Web & HTML
# Loader	Use When...	Code
# WebBaseLoader	Load content from a URL	=> loader = WebBaseLoader("https://example.com")
# UnstructuredHTMLLoader	Clean up raw HTML structure	=> loader = UnstructuredHTMLLoader("file.html")

# Load PDF with Tables using PDFPlumberLoader
# from langchain.document_loaders import PDFPlumberLoader
# loader = PDFPlumberLoader("report_with_tables.pdf")
# documents = loader.load()

# Load Excel Sheet using UnstructuredExcelLoader
# from langchain.document_loaders import UnstructuredExcelLoader
# loader = UnstructuredExcelLoader("financials.xlsx")
# documents = loader.load()

# Load JSON File
# from langchain.document_loaders import JSONLoader
# loader = JSONLoader(file_path="data.json", jq_schema=".records[].summary", text_content=True)
# documents = loader.load()

# Load DOCX File
# from langchain.document_loaders import UnstructuredWordDocumentLoader
# loader = UnstructuredWordDocumentLoader("policy.docx")
# documents = loader.load()



# How to Choose the Right Loader?
# | Content Type               | Recommended Loader                                          | Why                           |
# | -------------------------- | ----------------------------------------------------------- | ----------------------------- |
# | Simple PDFs                | `PyPDFLoader`                                               | Fast, reliable                |
# | PDFs with tables           | `PDFPlumberLoader`                                          | Can extract table content     |
# | PDFs with images / scanned | `UnstructuredPDFLoader`                                     | Uses OCR and layout modeling  |
# | Word / Excel               | `UnstructuredWordDocumentLoader`, `UnstructuredExcelLoader` | Best structure preservation   |
# | CSV / JSON                 | `PandasCSVLoader`, `JSONLoader`                             | Allows flexible preprocessing |

In [22]:
documents

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-08-26T06:56:00+00:00', 'author': 'hr', 'moddate': '2020-08-26T06:56:00+00:00', 'source': 'docs/company_policy.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}, page_content='SPIL Corporate HR Policies  \n \n \nSIRCA PAINTS INDIA LTD \nNEW DELHI  \n \n \n \n \nCORPORATE  \n  HUMAN RESOURCES \nPOLICIES & MANUALS'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-08-26T06:56:00+00:00', 'author': 'hr', 'moddate': '2020-08-26T06:56:00+00:00', 'source': 'docs/company_policy.pdf', 'total_pages': 24, 'page': 1, 'page_label': '2'}, page_content='SPIL Corporate HR Policies  \n \n \n \n \nSection 1: Introduction  \n \nThis handbook is the summary of the policies, procedures, guidance and benefits to the employees \nand organization. It is an introduction to our vision, mission, values, what you expect from us and \nwhat we 

In [23]:
# Step 3: Split PDF text into smaller chunks
# https://miro.medium.com/v2/resize:fit:1400/1*yfeUrFCr9oEVZofS8TvDEg.png
# RecursiveCharacterTextSplitter tries to split at logical boundaries (paragraph → sentence → word → character) — hence, recursive
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # chunk_size=500: Each chunk will be up to 500 characters long.
    chunk_overlap=100 # chunk_overlap=100: The last 100 characters of one chunk are repeated in the next chunk for context continuity.
)
chunks = text_splitter.split_documents(documents)

# Print number of chunks created
print(f"Splitted into {len(chunks)} chunks.")
print()

# Print first chunk for inspection
print(chunks[0])

Splitted into 122 chunks.

page_content='SPIL Corporate HR Policies  
 
 
SIRCA PAINTS INDIA LTD 
NEW DELHI  
 
 
 
 
CORPORATE  
  HUMAN RESOURCES 
POLICIES & MANUALS' metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-08-26T06:56:00+00:00', 'author': 'hr', 'moddate': '2020-08-26T06:56:00+00:00', 'source': 'docs/company_policy.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}


In [25]:
# Print second last chunk for inspection
print(chunks[-2])

page_content='SPIL Corporate HR Policies  
 
 
 
These consultants have to submit the invoice at end of the month and they are not entitled for the 
Provident Fund, ESIC and other statutory benefits.  
All other benefits of the company will depend upon the grades and hierarchy of the consultant. 
The same will be approved by the Managing Director at the time of joining of the consultant.  
 
Section : 20  Review and Amendment' metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-08-26T06:56:00+00:00', 'author': 'hr', 'moddate': '2020-08-26T06:56:00+00:00', 'source': 'docs/company_policy.pdf', 'total_pages': 24, 'page': 23, 'page_label': '24'}


In [24]:
# Print last chunk for inspection
print(chunks[-1])


page_content='Section : 20  Review and Amendment  
Management shall review this policy periodically and amendments required, if any shall be made 
accordingly.  
Section : 21 Residual Power 
This policy is basically guidelines and the management reserves the right to withdraw / modify to 
suit organization’s philosophy at any time without assigning any reason whatsoever. 
EFFECTIVE 
Commencement Of Policy  August 21, 2018  
 
 
 
Approved By : ___________SD/-_______________ 
Mr Sanjay Agarwal - CMD' metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-08-26T06:56:00+00:00', 'author': 'hr', 'moddate': '2020-08-26T06:56:00+00:00', 'source': 'docs/company_policy.pdf', 'total_pages': 24, 'page': 23, 'page_label': '24'}


In [26]:
# Why Is Chunking So Important in RAG?
# LLMs (like GPT-4) have a limited context window (e.g., 8K or 32K tokens), so you can't send the whole document.
# Chunking allows:
# -Semantic search over smaller pieces of the document
# -Better matching with the user's query
# -Faster retrieval, better accuracy

# Tradional vs Semantic Search example
# Traditional Search: Keyword-based search that looks for exact matches of words or phrases in documents.
# Semantic Search: Understands the meaning and context of the query to find relevant documents, even if they don't contain the exact keywords.
# Example:
# Query: "What is the company's leave policy after maternity leave?"
# Traditional Search Result: Might return documents that contain the exact phrase "maternity leave policy."
# Semantic Search Result: Would return documents that discuss leave policies, maternity benefits, and related topics, even if they don't use the exact phrase.

# Common Chunking Strategies (Used in Industry)
# | Splitter Class                          | Description                                           | Use Case                                           |
# | --------------------------------------- | ----------------------------------------------------- | -------------------------------------------------- |
# | `RecursiveCharacterTextSplitter`        | Splits at paragraphs → sentences → words → characters | Best for generic documents (PDFs, policies, books) |
# | `CharacterTextSplitter`                 | Splits at fixed character limits (naive)              | Simple logs, structured text                       |
# | `TokenTextSplitter`                     | Splits by token count (uses tokenizer like tiktoken)  | Precise control for GPT-3.5/4                      |
# | `SentenceTransformersTokenTextSplitter` | Aware of sentence boundaries + tokens                 | Best for multilingual and NLP-heavy documents      |
# | `MarkdownHeaderTextSplitter`            | Splits based on Markdown headers                      | Technical docs, blog posts, notebooks              |
# | `HTMLHeaderTextSplitter`                | Splits based on HTML tags                             | Websites, web-scraped data                         |
# | `Language` Splitters                    | Code-aware (Python, JS, etc.)                         | Splits by function/class — great for code docs     |


# How to Choose chunk_size and chunk_overlap?
# chunk_size: How big each chunk is (in characters or tokens)
# | Scenario                       | Recommended Chunk Size               |
# | ------------------------------ | ------------------------------------ |
# | Short emails, chat transcripts | 300–500 chars                        |
# | Policies, contracts, PDFs      | 500–1000 chars                       |
# | Technical manuals or FAQs      | 800–1500 chars                       |
# | Code or JSON files             | 100–300 chars (or by function/class) |
# | Scientific papers (dense info) | 1000–1500 chars or 256–512 tokens    |
# | For OpenAI GPT-4 (8k model)    | Chunk size ≤ 1000 tokens             |
# | For GPT-4 Turbo (128k model)   | Chunk size ≤ 3000 tokens             |

# chunk_overlap: How much to repeat from previous chunk
# | Purpose                              | Suggested Overlap                                      |
# | ------------------------------------ | ------------------------------------------------------ |
# | Maintain sentence continuity         | 10–20% of chunk\_size (e.g., 100 chars for 500 chunks) |
# | Prevent cutoff of entity names       | 100–150 chars                                          |
# | Use with semantic search (retriever) | 10–20%                                                 |
# | Use with QA or summarization         | 100–200 chars                                          |

# Example Best Practices:
# Contract Documents:
# RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# Contracts often have long clauses — more overlap helps preserve clause boundaries.

# FAQs or Web Pages:
# RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)

# Code Files:
# PythonCodeTextSplitter(chunk_lines=30, chunk_overlap=10)